In [1]:
import time

In [2]:
start_notebook = time.time()

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
!nvidia-smi

Mon Jan  5 16:44:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             50W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

# 1. Load Environment

In [5]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl

In [6]:
import transformers
import datasets
import accelerate
import peft
import bitsandbytes
import trl

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("trl:", trl.__version__)

transformers: 4.57.3
datasets: 4.0.0
accelerate: 1.12.0
peft: 0.18.0
bitsandbytes: 0.49.0
trl: 0.26.2


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 2. Import Libraries

In [8]:
import numpy as np
from datasets import Dataset, load_dataset, load_from_disk
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score

# 3. Load Dataset

In [9]:
name_dataset = 'IMDB'

In [10]:
dataset = load_dataset("imdb")

In [11]:
dataset.shape

{'train': (25000, 2), 'test': (25000, 2), 'unsupervised': (50000, 2)}

In [12]:
dataset_train = dataset['train']
pre_dataset_test = dataset['test']

In [13]:
split_test = pre_dataset_test.train_test_split(test_size=0.5, stratify_by_column = 'label', seed=42)

In [14]:
dataset_val = split_test['train']
dataset_test = split_test['test']

In [15]:
df_train = dataset_train.to_pandas()
df_val = dataset_val.to_pandas()
df_test = dataset_test.to_pandas()

**a. Analysis: Train Set**

In [16]:
df_train.shape

(25000, 2)

In [17]:
df_train['label'].value_counts()

,count
label,
0,12500
1,12500


In [18]:
round(df_train['label'].value_counts(normalize = True)*100, 2)

,proportion
label,
0,50.0
1,50.0


**b. Analysis: Validation Set**

In [19]:
df_val.shape

(12500, 2)

In [20]:
df_val['label'].value_counts()

,count
label,
1,6250
0,6250


In [21]:
round(df_val['label'].value_counts(normalize = True)*100, 2)

,proportion
label,
1,50.0
0,50.0


**c. Analysis: Test Set**

In [22]:
df_test.shape

(12500, 2)

In [23]:
df_test['label'].value_counts()

,count
label,
1,6250
0,6250


In [24]:
round(df_test['label'].value_counts(normalize = True)*100, 2)

,proportion
label,
1,50.0
0,50.0


**d. Save dataframes**

In [25]:
path_save = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/01.Datasets_Creation/{name_dataset}'

In [26]:
df_train.to_csv(f'{path_save}/df_train.csv')
df_val.to_csv(f'{path_save}/df_val.csv')
df_test.to_csv(f'{path_save}/df_test.csv')

# 4. BERT

In [27]:
name_model = "bert-base-uncased"

In [28]:
tokenizer = AutoTokenizer.from_pretrained(name_model)

In [29]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation = True)

In [30]:
tokenized_train = dataset_train.map(preprocess_function, batched = True)

In [31]:
tokenized_val = dataset_val.map(preprocess_function, batched = True)

In [32]:
tokenized_test = dataset_test.map(preprocess_function, batched = True)

Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

In [33]:
tokenized_train.save_to_disk(f'{path_save}/{name_model}/train')

Saving the dataset (0/1 shards):   0%|          | 0/25000 [00:00<?, ? examples/s]

In [34]:
tokenized_val.save_to_disk(f'{path_save}/{name_model}/val')

Saving the dataset (0/1 shards):   0%|          | 0/12500 [00:00<?, ? examples/s]

In [35]:
tokenized_test.save_to_disk(f'{path_save}/{name_model}/test')

Saving the dataset (0/1 shards):   0%|          | 0/12500 [00:00<?, ? examples/s]

# 5. DistilBERT

In [36]:
name_model = "distilbert-base-uncased"

In [37]:
tokenizer = AutoTokenizer.from_pretrained(name_model)

In [38]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation = True)

In [39]:
tokenized_train = dataset_train.map(preprocess_function, batched = True)

In [40]:
tokenized_val = dataset_val.map(preprocess_function, batched = True)

In [41]:
tokenized_test = dataset_test.map(preprocess_function, batched = True)

Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

In [42]:
tokenized_train.save_to_disk(f'{path_save}/{name_model}/train')

Saving the dataset (0/1 shards):   0%|          | 0/25000 [00:00<?, ? examples/s]

In [43]:
tokenized_val.save_to_disk(f'{path_save}/{name_model}/val')

Saving the dataset (0/1 shards):   0%|          | 0/12500 [00:00<?, ? examples/s]

In [44]:
tokenized_test.save_to_disk(f'{path_save}/{name_model}/test')

Saving the dataset (0/1 shards):   0%|          | 0/12500 [00:00<?, ? examples/s]

# 6. RoBERTa

In [45]:
name_model = "roberta-base"

In [46]:
tokenizer = AutoTokenizer.from_pretrained(name_model)

In [47]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation = True)

In [48]:
tokenized_train = dataset_train.map(preprocess_function, batched = True)

In [49]:
tokenized_val = dataset_val.map(preprocess_function, batched = True)

In [50]:
tokenized_test = dataset_test.map(preprocess_function, batched = True)

Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

In [51]:
tokenized_train.save_to_disk(f'{path_save}/{name_model}/train')

Saving the dataset (0/1 shards):   0%|          | 0/25000 [00:00<?, ? examples/s]

In [52]:
tokenized_val.save_to_disk(f'{path_save}/{name_model}/val')

Saving the dataset (0/1 shards):   0%|          | 0/12500 [00:00<?, ? examples/s]

In [53]:
tokenized_test.save_to_disk(f'{path_save}/{name_model}/test')

Saving the dataset (0/1 shards):   0%|          | 0/12500 [00:00<?, ? examples/s]

# 7. Execution Time

In [54]:
end_notebook = time.time()

In [55]:
delta_notebook = end_notebook - start_notebook
hours_notebook, rem_notebook = divmod(delta_notebook, 3600)
minutes_notebook, seconds_notebook = divmod(rem_notebook, 60)

print(f"Execution Notebook: {int(hours_notebook)}h {int(minutes_notebook)}m {seconds_notebook:.2f}s")

Execution Notebook: 0h 0m 42.99s
